# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and analyzing the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Name:", metadata.name)
print("Description:", metadata.description)

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets in the dataset
record_sets = dataset.metadata.record_sets
print("Record sets and @ids:")
for rs in record_sets:
    print(f"- Name: {rs.name} | @id: {rs.id}")

# Display the fields and columns for each record set
for rs in record_sets:
    fields = rs.fields
    print(f"\nRecordSet '{rs.name}' (@id: {rs.id}):")
    for field in fields:
        print(f"  Field: {field.name} | @id: {field.id} | Data Type: {field.data_type}")
        # Show columns if available
        if hasattr(field, 'columns') and field.columns:
            for col in field.columns:
                print(f"      Column: {col.name} | @id: {col.id} | Data Type: {col.data_type}")

## 3. Data Extraction
Load data from all record sets into DataFrames for analysis. Each record set and data field is referenced by its `@id`.

In [ ]:
# Use the discovered record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

dataframes = {}
for rec_id in record_set_ids:
    records = list(dataset.records(record_set=rec_id))
    df = pd.DataFrame(records)
    dataframes[rec_id] = df
    print(f"RecordSet @id: {rec_id}, columns: {df.columns.tolist()}")

# Preview the first record set
if record_set_ids:
    preview_rs_id = record_set_ids[0]
    print(f"Preview of RecordSet @id: {preview_rs_id}")
    display(dataframes[preview_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply typical data processing steps: filtering records, normalizing numeric fields, grouping records, and preparing for analysis. Reference fields by their `@id`.

In [ ]:
# Let's pick the first record set for demonstration
rs_id = record_set_ids[0]
df = dataframes[rs_id]

# Find numeric fields for filtering/normalization
numeric_fields = [col for col in df.columns if df[col].dtype.kind in 'fi']
print("Numeric fields:", numeric_fields)

# Select a numeric field, e.g. 'age_at_diagnosis' (replace with actual @id from the schema)
if numeric_fields:
    numeric_field = numeric_fields[0]
    threshold = df[numeric_field].median() if df[numeric_field].notna().sum() > 0 else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())
    
    # Normalize selected numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by another field (e.g., 'infertility')
    group_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
    if group_fields:
        group_field = group_fields[0]
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())

## 5. Visualization
Visualize numeric distributions and relationships between selected fields.

In [ ]:
# Visualize distribution for the selected numeric field
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_fields:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # If there are multiple numeric fields, show relationships
    if len(numeric_fields) >= 2:
        plt.figure(figsize=(6,4))
        sns.scatterplot(x=df[numeric_fields[0]], y=df[numeric_fields[1]], hue=df[group_fields[0]] if group_fields else None)
        plt.title(f"Relationship between {numeric_fields[0]} and {numeric_fields[1]}")
        plt.xlabel(numeric_fields[0])
        plt.ylabel(numeric_fields[1])
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load, inspect, and process clinical and genetic data from a FAIR² dataset using `mlcroissant`. Key steps included referencing entities by their `@id`, performing basic EDA, filtering and normalizing numeric fields, and visualizing distributions. Due to the dataset's small sample size and clinical specificity, results are illustrative and mostly exploratory. For more advanced analysis, consult the data limitations noted in the dataset metadata.